# 量子計算 × 機器學習完整教學 Notebook（Qiskit Aer）

> 這份 notebook 會「非常詳細」地解釋：
> - 什麼是 `make_moons`，為什麼常用於分類
> - classical data / quantum data 的差異
> - classical algo / quantum algo 的差異
> - 四種搭配組合如何評估
> - quantum algo 額外測試：**QSVM、QNN、QSVT（教學可執行近似版）**

---

## 你會得到什麼

1. **Classical Data + Classical Algo**（基準）
2. **Classical Data + Quantum Algo**（QSVM / QNN / QSVT-like）
3. **Quantum Data + Classical Algo**
4. **Quantum Data + Quantum Algo**（QSVM / QNN / QSVT-like）

每一步都附上中文註解，且特別補強「特殊寫法」的解釋。

## 0) 安裝（若環境尚未安裝）

> 你可以先執行這格，避免後續 import 失敗。

In [ ]:
# 若尚未安裝請取消註解
# %pip install qiskit qiskit-aer qiskit-machine-learning scikit-learn scipy matplotlib pandas

## 1) 匯入套件與全域設定（逐行解釋）

下面會看到一些你可能覺得「特殊」的寫法，我先說重點：

- `from ... import ...`：只把你要用的類別/函式載入，避免整個模組都寫長名稱。
- `np.random.seed(42)`：固定亂數，讓你每次跑出來的結果接近，方便教學比較。
- `AerSimulator`：量子電路模擬器，不需要真實量子硬體就可以跑。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- sklearn：經典機器學習工具箱 ---
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# --- Qiskit 核心與模擬器 ---
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

# --- Qiskit Machine Learning（本 notebook 的量子演算法主角）---
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_machine_learning.algorithms import QSVC, NeuralNetworkClassifier
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.neural_networks import EstimatorQNN

# --- 參數優化（給 QNN / 自訂模型使用）---
from scipy.optimize import minimize

# 固定隨機種子，讓實驗更可重現
SEED = 42
np.random.seed(SEED)
RNG = np.random.default_rng(SEED)

# 建立 Aer 模擬器
sim_qasm = AerSimulator(method="automatic")

## 2) `make_moons` 到底是什麼？為什麼常用？

`make_moons` 是 sklearn 內建的「玩具資料集產生器」，會產生兩群像「兩個月亮」的 2D 點。

### 為什麼它重要？

- 它不是線性可分（non-linear）
- 所以可以測試模型有沒有能力學到非線性邊界
- 非常適合拿來做「教學比較」

### 下面參數逐一解釋

- `n_samples=320`：樣本數 320 筆
- `noise=0.18`：加一點噪聲，讓資料比較像真實世界，不會太完美
- `random_state=SEED`：固定隨機，方便重現

In [ ]:
# 產生 classical data（經典資料）
X_classical, y_classical = make_moons(
    n_samples=320,
    noise=0.18,
    random_state=SEED,
)

print("X_classical shape:", X_classical.shape)  # (樣本數, 特徵數)
print("y_classical shape:", y_classical.shape)  # (樣本數,)

# 視覺化：你會看到兩個彎月形狀
plt.figure(figsize=(6, 5))
plt.scatter(X_classical[:, 0], X_classical[:, 1], c=y_classical, cmap="coolwarm", s=22)
plt.title("Classical Data from sklearn.make_moons")
plt.xlabel("feature x1")
plt.ylabel("feature x2")
plt.show()

## 3) 為什麼常看到 `train_test_split(..., stratify=y)` 這種寫法？

這個寫法也很多人會問，我在這裡補充：

- `test_size=0.30`：30% 當測試集
- `stratify=y`：分割時維持每一類標籤比例接近原始資料（很重要）
- `random_state=SEED`：固定分割結果

如果不加 `stratify=y`，有機會出現訓練/測試類別比例失衡，影響評估公平性。

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_classical,
    y_classical,
    test_size=0.30,
    random_state=SEED,
    stratify=y_classical,
)

print("Classical train/test:", Xc_train.shape, Xc_test.shape)

## 4) Classical Algorithm 基準：Logistic Regression（詳細註解）

這裡使用 `Pipeline([("scaler", ...), ("lr", ...)])` 是一種標準寫法：

- `StandardScaler()`：先把每個特徵標準化（平均 0、標準差 1）
- `LogisticRegression(...)`：再做分類

### 為什麼要 Pipeline？

它可以把「前處理 + 模型」綁成一個流程，避免資料洩漏與手動錯誤。

In [ ]:
clf_classical = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1200)),
])

clf_classical.fit(Xc_train, yc_train)
pred_cc = clf_classical.predict(Xc_test)
acc_cc = accuracy_score(yc_test, pred_cc)

print("[A] Classical Data + Classical Algo (Logistic Regression)")
print("Accuracy:", round(acc_cc, 4))
print(classification_report(yc_test, pred_cc, digits=4))

## 5) 產生 Quantum Data（詳細解釋）

你要求「classical data 與 quantum data 都要詳細模擬」。

這裡的 quantum data 來源不是人工公式，而是：

1. 隨機取參數 \(\alpha, \beta\)
2. 把參數送進量子電路
3. 用 Aer 模擬量測得到 bitstring 機率 `p00,p01,p10,p11`
4. 這 4 個機率當成特徵
5. 再用量子態中的 \(<Z>\) 決定標籤

因此資料「本質上」是由量子過程生成。

In [ ]:
def quantum_data_circuit(alpha, beta):
    """建立資料來源量子電路。"""
    qc = QuantumCircuit(2)
    qc.ry(alpha, 0)
    qc.rx(beta, 1)
    qc.cz(0, 1)
    qc.ry(0.5 * alpha, 1)
    return qc


def z_expectation_on_qubit0(statevec_2q):
    """
    計算 qubit-0 的 Z 期望值。
    對 2-qubit（|00>,|01>,|10>,|11>）對應 z 值 [+1,-1,+1,-1]。
    """
    probs = np.abs(statevec_2q) ** 2
    zvals = np.array([+1, -1, +1, -1])
    return float(np.dot(probs, zvals))


def measure_probs(alpha, beta, shots=512):
    """回傳 [p00,p01,p10,p11]。"""
    qc = quantum_data_circuit(alpha, beta)
    qcm = qc.copy()
    qcm.measure_all()
    counts = sim_qasm.run(qcm, shots=shots).result().get_counts()

    # 注意：Qiskit 顯示 bitstring 順序為 q1q0
    ordered = ["00", "01", "10", "11"]
    probs = np.array([counts.get(k, 0) / shots for k in ordered], dtype=float)
    return probs


def quantum_label(alpha, beta):
    """以 <Z0> >= 0 當 label=1，否則 0。"""
    st = Statevector.from_instruction(quantum_data_circuit(alpha, beta)).data
    return int(z_expectation_on_qubit0(st) >= 0.0)


def generate_quantum_dataset(n_samples=320, shots=512, seed=SEED):
    rng = np.random.default_rng(seed)
    Xq, yq = [], []
    for _ in range(n_samples):
        alpha = rng.uniform(-np.pi, np.pi)
        beta = rng.uniform(-np.pi, np.pi)
        Xq.append(measure_probs(alpha, beta, shots=shots))
        yq.append(quantum_label(alpha, beta))
    return np.array(Xq), np.array(yq)

In [ ]:
X_quantum, y_quantum = generate_quantum_dataset(n_samples=320, shots=512, seed=SEED)

Xq_train, Xq_test, yq_train, yq_test = train_test_split(
    X_quantum,
    y_quantum,
    test_size=0.30,
    random_state=SEED,
    stratify=y_quantum,
)

print("X_quantum shape:", X_quantum.shape)   # (320, 4)
print("y_quantum shape:", y_quantum.shape)
print("Quantum train/test:", Xq_train.shape, Xq_test.shape)

## 6) Quantum Algorithm #1：QSVM（Quantum Support Vector Machine）

### QSVM 在做什麼？

- 傳統 SVM 會用 kernel 測「樣本相似度」
- QSVM 把 kernel 改成量子特徵映射後得到的量子 kernel
- 直覺：用量子電路把資料映射到量子特徵空間，再比相似度

### 這裡的特殊寫法

- `ZZFeatureMap(feature_dimension=..., reps=2)`：把資料編碼進可含糾纏的特徵電路
- `FidelityQuantumKernel(...)`：根據 state fidelity 建 kernel
- `QSVC(...)`：吃 kernel 做 SVM 分類

In [ ]:
def fit_eval_qsvm(X_train, y_train, X_test, y_test, reps=2):
    feature_map = ZZFeatureMap(feature_dimension=X_train.shape[1], reps=reps)
    qkernel = FidelityQuantumKernel(feature_map=feature_map)
    model = QSVC(quantum_kernel=qkernel)

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    acc = accuracy_score(y_test, pred)
    return model, pred, acc

## 7) Quantum Algorithm #2：QNN（Quantum Neural Network）

### QNN 在做什麼？

- 結構很像「可訓練量子電路 + classical optimizer」
- `EstimatorQNN` 會把可觀測量期望值當作模型輸出
- `NeuralNetworkClassifier` 幫你包裝成 sklearn 風格分類器

### 特別注意

- QNN 對資料尺度很敏感，所以要把輸入壓到角度範圍（例如 \([-\pi, \pi]\)）

In [ ]:
def scale_to_angle(X):
    """把輸入特徵縮放到 [-pi, pi]，降低角度爆炸。"""
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    Xang = np.clip(Xs, -2, 2) * (np.pi / 2)
    return Xang, scaler


def fit_eval_qnn(X_train, y_train, X_test, y_test, reps_feature=1, reps_ansatz=1):
    # 1) 角度縮放
    X_train_ang, scaler = scale_to_angle(X_train)
    X_test_ang = np.clip(scaler.transform(X_test), -2, 2) * (np.pi / 2)

    # 2) 特徵映射 + 可訓練 ansatz
    n_features = X_train_ang.shape[1]
    feature_map = ZZFeatureMap(feature_dimension=n_features, reps=reps_feature)
    ansatz = RealAmplitudes(num_qubits=n_features, reps=reps_ansatz)

    # 3) 合併成單一電路（先編碼再訓練）
    qc = QuantumCircuit(n_features)
    qc.compose(feature_map, inplace=True)
    qc.compose(ansatz, inplace=True)

    # 4) EstimatorQNN：將電路轉成可訓練神經網路
    qnn = EstimatorQNN(
        circuit=qc,
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
    )

    # 5) 用 NeuralNetworkClassifier 包裝 + COBYLA 優化
    clf = NeuralNetworkClassifier(
        neural_network=qnn,
        optimizer=minimize,
        initial_point=RNG.normal(0, 0.1, size=len(ansatz.parameters)),
    )

    clf.fit(X_train_ang, y_train)
    pred = clf.predict(X_test_ang)
    acc = accuracy_score(y_test, pred)
    return clf, pred, acc

## 8) Quantum Algorithm #3：QSVT（教學可執行近似版）

你指定要測 `QSVT`。這裡我先說清楚：

- 嚴格的 QSVT（Quantum Singular Value Transformation）通常需要 block-encoding 等較進階構造。
- 在教學 notebook 中，我們做**可執行近似流程**：
  1. 先用量子特徵映射得到 kernel 相似度
  2. 對相似度套用「奇次多項式濾波」\(p(k)=a_1 k + a_3 k^3\)
  3. 這一步對應 QSVT 中「以多項式轉換光譜」的核心概念
  4. 最後用閾值判斷分類

> 這是 **QSVT-like 教學模擬**，目標是理解概念與流程，不宣稱等價於完整容錯級 QSVT 實作。

In [ ]:
def qsvt_like_train(X_train, y_train, reps=2):
    """
    QSVT-like：
    - 用 quantum kernel 建每個樣本到 class-1 prototype 的相似度 k
    - 學習奇次多項式 p(k)=a1*k + a3*k^3
    - 以 BCE 作為損失
    """
    feature_map = ZZFeatureMap(feature_dimension=X_train.shape[1], reps=reps)
    qkernel = FidelityQuantumKernel(feature_map=feature_map)

    # class-1 prototype（簡化做法：正類平均向量）
    proto = X_train[y_train == 1].mean(axis=0, keepdims=True)

    # 計算每筆資料對 prototype 的量子 kernel 相似度
    K = qkernel.evaluate(x_vec=X_train, y_vec=proto).reshape(-1)

    def sigmoid(z):
        return 1 / (1 + np.exp(-z))

    def bce(y, p, eps=1e-9):
        p = np.clip(p, eps, 1 - eps)
        return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

    def objective(params):
        a1, a3, b = params
        poly = a1 * K + a3 * (K ** 3) + b
        prob = sigmoid(poly)
        return bce(y_train, prob)

    res = minimize(objective, x0=np.array([1.0, 0.1, 0.0]), method="COBYLA", options={"maxiter": 120})

    model = {
        "qkernel": qkernel,
        "proto": proto,
        "params": res.x,
        "train_loss": float(res.fun),
    }
    return model


def qsvt_like_predict(model, X):
    K = model["qkernel"].evaluate(x_vec=X, y_vec=model["proto"]).reshape(-1)
    a1, a3, b = model["params"]
    logits = a1 * K + a3 * (K ** 3) + b
    prob = 1 / (1 + np.exp(-logits))
    pred = (prob >= 0.5).astype(int)
    return pred, prob

## 9) 在 Classical Data 上測三種 Quantum Algo

這裡你會看到：
- Classical data + QSVM
- Classical data + QNN
- Classical data + QSVT-like

In [ ]:
# 為了讓量子特徵映射輸入穩定，先做標準化
scaler_c = StandardScaler()
Xc_train_std = scaler_c.fit_transform(Xc_train)
Xc_test_std = scaler_c.transform(Xc_test)

# QSVM
qsvm_c, pred_c_qsvm, acc_c_qsvm = fit_eval_qsvm(Xc_train_std, yc_train, Xc_test_std, yc_test, reps=2)
print("[B1] Classical Data + QSVM Accuracy:", round(acc_c_qsvm, 4))

# QNN（若你的 qiskit-machine-learning 版本對 optimizer 介面不同，可能要調整）
try:
    qnn_c, pred_c_qnn, acc_c_qnn = fit_eval_qnn(Xc_train_std, yc_train, Xc_test_std, yc_test, reps_feature=1, reps_ansatz=1)
except Exception as e:
    qnn_c, pred_c_qnn, acc_c_qnn = None, None, np.nan
    print("[B2] QNN 此環境未成功執行，錯誤摘要：", str(e)[:200])
print("[B2] Classical Data + QNN Accuracy:", acc_c_qnn)

# QSVT-like
qsvt_c = qsvt_like_train(Xc_train_std, yc_train, reps=2)
pred_c_qsvt, prob_c_qsvt = qsvt_like_predict(qsvt_c, Xc_test_std)
acc_c_qsvt = accuracy_score(yc_test, pred_c_qsvt)
print("[B3] Classical Data + QSVT-like Accuracy:", round(acc_c_qsvt, 4))

## 10) 在 Quantum Data 上測：Classical Algo + 三種 Quantum Algo

先做 quantum data + classical algo（Logistic Regression），再做 quantum algo（QSVM / QNN / QSVT-like）。

In [ ]:
# [C] Quantum Data + Classical Algo
clf_qc = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=1200)),
])
clf_qc.fit(Xq_train, yq_train)
pred_q_classical = clf_qc.predict(Xq_test)
acc_q_classical = accuracy_score(yq_test, pred_q_classical)
print("[C] Quantum Data + Classical Algo Accuracy:", round(acc_q_classical, 4))

# 量子模型輸入標準化
scaler_q = StandardScaler()
Xq_train_std = scaler_q.fit_transform(Xq_train)
Xq_test_std = scaler_q.transform(Xq_test)

# [D1] Quantum Data + QSVM
qsvm_q, pred_q_qsvm, acc_q_qsvm = fit_eval_qsvm(Xq_train_std, yq_train, Xq_test_std, yq_test, reps=2)
print("[D1] Quantum Data + QSVM Accuracy:", round(acc_q_qsvm, 4))

# [D2] Quantum Data + QNN
try:
    qnn_q, pred_q_qnn, acc_q_qnn = fit_eval_qnn(Xq_train_std, yq_train, Xq_test_std, yq_test, reps_feature=1, reps_ansatz=1)
except Exception as e:
    qnn_q, pred_q_qnn, acc_q_qnn = None, None, np.nan
    print("[D2] QNN 此環境未成功執行，錯誤摘要：", str(e)[:200])
print("[D2] Quantum Data + QNN Accuracy:", acc_q_qnn)

# [D3] Quantum Data + QSVT-like
qsvt_q = qsvt_like_train(Xq_train_std, yq_train, reps=2)
pred_q_qsvt, prob_q_qsvt = qsvt_like_predict(qsvt_q, Xq_test_std)
acc_q_qsvt = accuracy_score(yq_test, pred_q_qsvt)
print("[D3] Quantum Data + QSVT-like Accuracy:", round(acc_q_qsvt, 4))

## 11) 統整結果表（四種搭配 + quantum algo 細分）

In [ ]:
summary = pd.DataFrame([
    {"Setting": "A: Classical Data + Classical Algo (LR)", "Accuracy": acc_cc},
    {"Setting": "B1: Classical Data + QSVM", "Accuracy": acc_c_qsvm},
    {"Setting": "B2: Classical Data + QNN", "Accuracy": acc_c_qnn},
    {"Setting": "B3: Classical Data + QSVT-like", "Accuracy": acc_c_qsvt},
    {"Setting": "C: Quantum Data + Classical Algo (LR)", "Accuracy": acc_q_classical},
    {"Setting": "D1: Quantum Data + QSVM", "Accuracy": acc_q_qsvm},
    {"Setting": "D2: Quantum Data + QNN", "Accuracy": acc_q_qnn},
    {"Setting": "D3: Quantum Data + QSVT-like", "Accuracy": acc_q_qsvt},
]).sort_values("Accuracy", ascending=False)

summary

## 12) 每一段「特殊寫法」速查（你可以回來對照）

1. `make_moons(...)`：產生非線性二元分類資料。
2. `train_test_split(..., stratify=y)`：維持類別比例。
3. `Pipeline([...])`：把前處理和模型打包，避免漏步驟。
4. `ZZFeatureMap`：把 classical feature 轉成量子電路特徵映射。
5. `FidelityQuantumKernel`：用量子態重疊度建立 kernel。
6. `QSVC`：在量子 kernel 上跑 SVM。
7. `EstimatorQNN + NeuralNetworkClassifier`：把參數化量子電路當神經網路訓練。
8. `QSVT-like p(k)=a1*k+a3*k^3`：示範「多項式光譜轉換」核心概念。

---

如果你希望，我下一版可以再補：
- 每個模型的決策邊界圖（2D）
- confusion matrix
- 執行時間比較（訓練秒數）
- 不同 `shots`、`reps`、`noise` 的敏感度分析